In [ ]:
# ================================================================
# 🇹🇳 WHISPER TUNISIEN - GOOGLE COLAB
# VERSION CORRIGÉE
#
# Modèle :
# oddadmix/Whisperv3-tunisian-codeswitch
#
# ✅ Microphone Colab
# ✅ Upload WAV / MP3 / M4A / WEBM
# ✅ Conversion automatique 16 kHz mono
# ✅ Aucun conflit FFmpeg input/output
# ✅ Lecture de l'audio
# ✅ GPU T4
# ✅ Tunisien + français + anglais
# ✅ Sans Gradio
# ================================================================


# ================================================================
# 0. INSTALLATION
# ================================================================

!pip install -q -U transformers accelerate librosa soundfile


# ================================================================
# 1. IMPORTS
# ================================================================

import os
import uuid
import subprocess
import torch

from base64 import b64decode
from google.colab import output, files
from IPython.display import Audio, display

from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline
)


# ================================================================
# 2. CONFIGURATION
# ================================================================

MODEL_ID = "oddadmix/Whisperv3-tunisian-codeswitch"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1

DTYPE = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)


print("=" * 70)
print("🇹🇳 WHISPER TUNISIEN")
print("=" * 70)

print("Model :", MODEL_ID)
print("Device:", DEVICE)

if torch.cuda.is_available():

    print("GPU   :", torch.cuda.get_device_name(0))

    vram = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print("VRAM  :", round(vram, 2), "GB")

else:

    print("\n⚠️ GPU non détecté.")
    print("Colab > Runtime > Change runtime type > T4 GPU")


# ================================================================
# 3. CHARGEMENT PROCESSOR
# ================================================================

print("\n⏳ Chargement processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

print("✅ Processor chargé")


# ================================================================
# 4. CHARGEMENT MODELE
# ================================================================

print("\n⏳ Chargement Whisper tunisien...")
print("Le modèle fait environ 6 GB.\n")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    low_cpu_mem_usage=True
)

model.to(DEVICE)
model.eval()

print("✅ Modèle chargé")


# ================================================================
# 5. PIPELINE ASR
# ================================================================

asr = pipeline(
    task="automatic-speech-recognition",

    model=model,

    tokenizer=processor.tokenizer,

    feature_extractor=processor.feature_extractor,

    dtype=DTYPE,

    device=PIPELINE_DEVICE
)

print("✅ Pipeline ASR prêt")


# ================================================================
# 6. CONVERSION AUDIO
#
# IMPORTANT :
# Le fichier de sortie a toujours un nouveau nom.
# Donc FFmpeg ne peut jamais écraser le fichier source.
# ================================================================

def convert_to_whisper_wav(input_path):

    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Fichier introuvable : {input_path}"
        )

    # Nouveau nom unique à chaque conversion
    unique_id = uuid.uuid4().hex[:8]

    output_path = (
        f"/content/whisper_ready_{unique_id}.wav"
    )

    command = [
        "ffmpeg",
        "-y",

        "-i",
        input_path,

        # pas de vidéo
        "-vn",

        # mono
        "-ac",
        "1",

        # 16 kHz
        "-ar",
        "16000",

        # WAV PCM 16-bit
        "-acodec",
        "pcm_s16le",

        output_path
    ]

    process = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if process.returncode != 0:

        error = process.stderr.decode(
            "utf-8",
            errors="ignore"
        )

        raise RuntimeError(
            "Erreur FFmpeg :\n" + error
        )

    if not os.path.exists(output_path):

        raise RuntimeError(
            "FFmpeg n'a pas créé le fichier WAV."
        )

    return output_path


# ================================================================
# 7. TRANSCRIPTION
# ================================================================

def transcribe(audio_path):

    if audio_path is None:
        print("❌ Aucun audio fourni.")
        return None

    if not os.path.exists(audio_path):
        print("❌ Fichier introuvable :", audio_path)
        return None


    print("\n")
    print("=" * 70)
    print("🎧 AUDIO")
    print("=" * 70)

    print("Fichier source :", audio_path)


    # ------------------------------------------------------------
    # Conversion vers format Whisper
    # ------------------------------------------------------------

    print("\n🔄 Préparation WAV 16 kHz mono...")

    try:

        whisper_audio = convert_to_whisper_wav(
            audio_path
        )

    except Exception as e:

        print("\n❌ Erreur conversion audio :")
        print(e)

        return None


    print("✅ Fichier Whisper :", whisper_audio)


    # ------------------------------------------------------------
    # Lecture audio
    # ------------------------------------------------------------

    print("\n🔊 Audio envoyé au modèle :")

    display(
        Audio(
            whisper_audio,
            autoplay=False
        )
    )


    # ------------------------------------------------------------
    # Transcription
    # ------------------------------------------------------------

    print("\n⏳ Transcription en cours...")


    try:

        with torch.inference_mode():

            result = asr(
                whisper_audio,

                generate_kwargs={
                    "language": "ar",
                    "task": "transcribe"
                },

                return_timestamps=False
            )


        text = result["text"].strip()


        print("\n")
        print("=" * 70)
        print("🇹🇳 TRANSCRIPTION")
        print("=" * 70)

        print()
        print(text)
        print()

        print("=" * 70)


        return text


    except Exception as e:

        print("\n❌ Erreur transcription :")
        print(str(e))

        return None


# ================================================================
# 8. JAVASCRIPT POUR MICROPHONE
# ================================================================

RECORD_JS = """

const sleep = time =>
    new Promise(resolve =>
        setTimeout(resolve, time)
    );


const blobToBase64 = blob =>
    new Promise(resolve => {

        const reader = new FileReader();

        reader.onloadend = () =>
            resolve(reader.result);

        reader.readAsDataURL(blob);
    });


async function recordAudio(seconds) {

    const stream =
        await navigator.mediaDevices.getUserMedia({
            audio: true
        });


    const recorder =
        new MediaRecorder(stream);


    let chunks = [];


    recorder.ondataavailable = event => {

        if (event.data.size > 0) {
            chunks.push(event.data);
        }

    };


    recorder.start();


    await sleep(
        seconds * 1000
    );


    recorder.stop();


    await new Promise(resolve => {
        recorder.onstop = resolve;
    });


    stream
        .getTracks()
        .forEach(track => track.stop());


    const blob = new Blob(
        chunks,
        {
            type: "audio/webm"
        }
    );


    return await blobToBase64(blob);
}

"""


# ================================================================
# 9. ENREGISTREMENT MICROPHONE
#
# IMPORTANT :
# Ici on retourne le WEBM ORIGINAL.
# On ne le convertit PAS ici.
#
# La conversion sera faite UNE SEULE FOIS
# dans transcribe().
# ================================================================

def record_audio(seconds=10):

    print("\n")
    print("=" * 70)
    print("🎙️ ENREGISTREMENT MICROPHONE")
    print("=" * 70)

    print(
        f"\n🎙️ Parle pendant {seconds} secondes."
    )

    print(
        "Autorise le microphone si Chrome le demande."
    )

    print("\n🔴 Enregistrement...")


    try:

        audio_data = output.eval_js(
            RECORD_JS
            +
            f"\nrecordAudio({seconds});"
        )


        if audio_data is None:
            raise RuntimeError(
                "Aucune donnée audio reçue."
            )


        audio_binary = b64decode(
            audio_data.split(",")[1]
        )


        unique_id = uuid.uuid4().hex[:8]

        webm_path = (
            f"/content/microphone_{unique_id}.webm"
        )


        with open(
            webm_path,
            "wb"
        ) as f:

            f.write(audio_binary)


        print("\n✅ Enregistrement terminé")

        print(
            "Fichier :",
            webm_path
        )


        # --------------------------------------------------------
        # Réécoute fichier original
        # --------------------------------------------------------

        print("\n🔊 Réécoute :")

        display(
            Audio(
                webm_path,
                autoplay=False
            )
        )


        return webm_path


    except Exception as e:

        print("\n❌ Erreur microphone :")
        print(str(e))

        print(
            "\nVérifie que le navigateur "
            "autorise l'accès au microphone."
        )

        return None


# ================================================================
# 10. MICROPHONE + TRANSCRIPTION
# ================================================================

def record_and_transcribe(seconds=10):

    # 1. Enregistrement WEBM
    audio_path = record_audio(
        seconds
    )

    if audio_path is None:
        return None


    # 2. Conversion + Whisper
    result = transcribe(
        audio_path
    )


    return result


# ================================================================
# 11. UPLOAD AUDIO
# ================================================================

def upload_audio():

    print("\n")
    print("=" * 70)
    print("📁 UPLOAD AUDIO")
    print("=" * 70)

    print(
        "\nFormats : WAV / MP3 / M4A / WEBM / OGG..."
    )


    uploaded = files.upload()


    if len(uploaded) == 0:

        print("❌ Aucun fichier.")

        return None


    filename = list(
        uploaded.keys()
    )[0]


    filepath = os.path.join(
        "/content",
        filename
    )


    print(
        "\n✅ Fichier chargé :",
        filepath
    )


    return filepath


# ================================================================
# 12. UPLOAD + TRANSCRIPTION
# ================================================================

def upload_and_transcribe():

    audio_path = upload_audio()

    if audio_path is None:
        return None


    return transcribe(
        audio_path
    )


# ================================================================
# 13. TEST D'UN FICHIER DEJA PRESENT
# ================================================================

def transcribe_file(path):

    return transcribe(
        path
    )


# ================================================================
# 14. INFORMATIONS FINALES
# ================================================================

print("\n")
print("=" * 70)
print("✅ WHISPER TUNISIEN PRÊT")
print("=" * 70)

print("""

🎙️ TEST MICROPHONE 10 secondes :

    record_and_transcribe(10)


🎙️ TEST MICROPHONE 20 secondes :

    record_and_transcribe(20)


📁 UPLOAD AUDIO :

    upload_and_transcribe()


📁 FICHIER DEJA PRESENT :

    transcribe_file("/content/mon_audio.wav")

""")


# ================================================================
# FIN
# ================================================================

🇹🇳 WHISPER TUNISIEN
Model : oddadmix/Whisperv3-tunisian-codeswitch
Device: cuda:0
GPU   : Tesla T4
VRAM  : 14.56 GB

⏳ Chargement processor...
✅ Processor chargé

⏳ Chargement Whisper tunisien...
Le modèle fait environ 6 GB.



Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

✅ Modèle chargé
✅ Pipeline ASR prêt


✅ WHISPER TUNISIEN PRÊT


🎙️ TEST MICROPHONE 10 secondes :

    record_and_transcribe(10)


🎙️ TEST MICROPHONE 20 secondes :

    record_and_transcribe(20)


📁 UPLOAD AUDIO :

    upload_and_transcribe()


📁 FICHIER DEJA PRESENT :

    transcribe_file("/content/mon_audio.wav")




In [ ]:
record_and_transcribe(20)



🎙️ ENREGISTREMENT MICROPHONE

🎙️ Parle pendant 20 secondes.
Autorise le microphone si Chrome le demande.

🔴 Enregistrement...

✅ Enregistrement terminé
Fichier : /content/microphone_a1d14a2f.webm

🔊 Réécoute :




🎧 AUDIO
Fichier source : /content/microphone_a1d14a2f.webm

🔄 Préparation WAV 16 kHz mono...
✅ Fichier Whisper : /content/whisper_ready_0361dcbb.wav

🔊 Audio envoyé au modèle :



⏳ Transcription en cours...


🇹🇳 TRANSCRIPTION

خوانا من يومك يلي تفاتي نجومك ماني باش نلومك ماني من غدرك موجوع سلام عليكم بالله نحب نعدي varilix trois point zero couleur



'خوانا من يومك يلي تفاتي نجومك ماني باش نلومك ماني من غدرك موجوع سلام عليكم بالله نحب نعدي varilix trois point zero couleur'